# Training (Thresholds)

This notebook contains the code used to perform the learning of a small number of individual hyperparameters of the complete tracking system. Each parameter is independently optimized using a grid search.

In [1]:
import sys

In [2]:
# Add the `code` directory to the path.
sys.path.append("../")

import kflearn

In [6]:
# This code is just here so I can reload the external files in case I make
# changes to them after having already imported them.
import importlib

importlib.reload(kflearn);

We now experiment with tuning different parameters of the system. The following will be tuned for this project:
* `threshold` in the detector. Higher values mean that the detector needs to be more sure about there being a person.
* `mo_threshold` in the tracker. Sets the threshold for when an observation is to be matched against an existing track.
* `mn_threshold` in the tracker. Sets the threshold for when two observation in different images not matched to any track should be associated.

In [3]:
def apply(value, _, detect, track):
    detect.threshold = value

kflearn.evaluate_params(
    [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9], apply, "..")

for 0.1...MPJPE 10.178, MOTA 0.67, P 0.790, R 0.926
for 0.2...MPJPE 7.403, MOTA 0.86, P 0.927, R 0.931
for 0.3...MPJPE 5.205, MOTA 0.89, P 0.972, R 0.920
for 0.4...MPJPE 5.166, MOTA 0.90, P 0.981, R 0.914
for 0.5...MPJPE 5.117, MOTA 0.90, P 0.984, R 0.913
for 0.6...MPJPE 4.960, MOTA 0.89, P 0.985, R 0.908
for 0.7...MPJPE 4.518, MOTA 0.89, P 0.987, R 0.900
for 0.8...MPJPE 4.468, MOTA 0.87, P 0.983, R 0.888
for 0.9...MPJPE 8.302, MOTA 0.58, P 0.891, R 0.648


Above we see that as expected, precision improves with higher values while recall favors smaller thresholds. Overall, the best MOTA is achieved with a threshold of 0.4 to 0.5. Since among them the value of 0.5 showed the lowest MPJPE, this value will be selected for the final evaluation. Note that this was already the default value used for the baseline.

In [ ]:
def apply(value, _, detect, track):
    track.mo_threshold = value

kflearn.evaluate_params([-10, -8, -6, -4, -2, 0, 2, 4, 6, 8], apply, "..")

for -10...MPJPE 4.938, MOTA -0.02, P 0.975, R 0.892
for -8...MPJPE 4.938, MOTA -0.02, P 0.975, R 0.892
for -6...MPJPE 14.337, MOTA 0.23, P 0.641, R 0.836
for -4...MPJPE 10.166, MOTA 0.77, P 0.879, R 0.904
for -2...MPJPE 5.583, MOTA 0.88, P 0.972, R 0.907
for 0...MPJPE 5.117, MOTA 0.90, P 0.984, R 0.913
for 2...MPJPE 5.023, MOTA 0.90, P 0.984, R 0.912
for 4...MPJPE 5.278, MOTA 0.88, P 0.976, R 0.905
for 6...MPJPE 5.280, MOTA 0.88, P 0.976, R 0.905
for 8...MPJPE 5.280, MOTA 0.88, P 0.976, R 0.905


An optimal value for `mo_threshold` seems to be between 0 and 2. Between the two the value of 2 has the lower MPJPE, so I choose if for the further evaluation.

In [ ]:
def apply(value, _, detect, track):
    track.mo_threshold = 2
    track.mn_threshold = value

kflearn.evaluate_params([-10, -8, -6, -4, -2, 0, 2, 4, 6, 8], apply, "..")

for -10...MPJPE 0.000, MOTA 0.00, P 0.000, R 0.000
for -8...MPJPE 3.878, MOTA 0.87, P 0.997, R 0.871
for -6...MPJPE 4.091, MOTA 0.91, P 0.995, R 0.912
for -4...MPJPE 4.256, MOTA 0.90, P 0.989, R 0.914
for -2...MPJPE 4.889, MOTA 0.90, P 0.985, R 0.913
for 0...MPJPE 5.023, MOTA 0.90, P 0.984, R 0.912
for 2...MPJPE 5.023, MOTA 0.90, P 0.984, R 0.912
for 4...MPJPE 5.023, MOTA 0.90, P 0.984, R 0.912
for 6...MPJPE 4.956, MOTA 0.90, P 0.984, R 0.912
for 8...MPJPE 4.956, MOTA 0.90, P 0.983, R 0.912


The system is quite robust to a large range of values for the `mn_threshold` parameter. A value of around -6 seems to get the best MOTA and MPJPE scores, and seems to be right at the edge of where performance starts to decrease sharply. A value of -6 will be used for the final evaluation.